# Building a Custom Chat Template on a Base Model

An *instruct* model ships with a chat template and special role tokens. A **base model** (here `mistralai/Mistral-7B-v0.3`) does not — it was only trained to continue raw text. To fine-tune a base model for chat-style tasks, we first have to **give it a chat format**.

This notebook shows how to:

1. Load the base model in **4-bit**.
2. Inspect the tokenizer and see that role markers like `<system>` are *not* single tokens.
3. **Add custom special tokens** (`<PAD>`, `<system>`, `<user>`, `<assistant>`).
4. **Resize the model's embedding table** so it has vectors for the new tokens.
5. Define a **custom chat template** and format the dataset with it.

> **Requirements:** a GPU runtime and a Hugging Face token (loaded from `.env`). Outputs below were captured from a real run.

## 1. Setup & authentication

In [ ]:
# --- Install dependencies ---
# transformers/datasets/bitsandbytes/trl/peft + python-dotenv for the token.
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load the Hugging Face token from a local .env file (see .env.example):
#   HF_TOKEN=hf_your_token_here
load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token or hf_token == "your_hugging_face_token_here":
    raise ValueError(
        "HF_TOKEN not found. Create a .env file with HF_TOKEN=hf_your_token_here "
        "(get a token at https://huggingface.co/settings/tokens)."
    )

# Authenticate so we can download gated models and push results to the Hub.
login(token=hf_token)
print("Successfully authenticated with the Hugging Face Hub.")

Successfully authenticated with the Hugging Face Hub.


## 2. Load a base model (4-bit)

Mistral 7B v0.3 is a *base* model, so it has no chat template of its own.

In [ ]:
# --- Load a BASE model (Mistral 7B v0.3) in 4-bit ---
# Unlike an instruct model, a base model has NO built-in chat template, so we
# will teach it our own chat format below. 4-bit keeps the 7B model on a small GPU.
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-v0.3"

config_4bit = BitsAndBytesConfig(load_in_4bit=True)

model_4bit = AutoModelForCausalLM.from_pretrained(

                                                  model_name,
                                                  quantization_config=config_4bit,
                                                  device_map="auto",
                                                  trust_remote_code=True
)

tokenizer =AutoTokenizer.from_pretrained(model_name,padding_side="left",trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
# Load the Docker NL->command dataset we'll format with the custom template.
from datasets import load_dataset

dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

## 3. Inspect the tokenizer's special tokens

In [ ]:
# The base tokenizer's special tokens: only <s>, </s>, <unk> -- no chat/role tokens.
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>']

## 4. The problem: role markers aren't single tokens yet

In [ ]:
# Our desired role markers aren't real tokens yet, so each splits into several
# sub-word pieces (multiple ids) -- inefficient and semantically weak.
tokens = ['<system>', '<user>', '<assistant>']

for token in tokens:
    token_id = tokenizer.encode(token,add_special_tokens=False)
    print(token,token_id)

<system> [1291, 7342, 29535]
<user> [1291, 2606, 29535]
<assistant> [1291, 1257, 11911, 29535]


In [ ]:
# A base model/tokenizer also has no dedicated padding token defined.
tokenizer.pad_token

## 5. Add custom special tokens

In [ ]:
# Register new special tokens: a pad token plus dedicated <system>/<user>/
# <assistant> role markers. add_special_tokens returns how many NEW tokens (4).
special_token = {

                 'pad_token': '<PAD>',
                 'additional_special_tokens': ['<system>', '<user>', '<assistant>']

}

tokenizer.add_special_tokens(special_token)

4

In [ ]:
# Now each role marker encodes to a SINGLE dedicated token id.
tokens = ['<system>', '<user>', '<assistant>','<PAD>']

for token in tokens:
    token_id = tokenizer.encode(token,add_special_tokens=False)
    print(token,token_id)

<system> [32769]
<user> [32770]
<assistant> [32771]
<PAD> [32768]


In [ ]:
# The tokenizer vocabulary grew by the 4 tokens we added (32768 -> 32772).
len(tokenizer)

32772

In [ ]:
# But the model's embedding table still expects the OLD vocab size (32768)...
model_4bit.config.vocab_size

32768

## 6. Resize the model's embeddings to match

In [ ]:
# ...so resize the model's token embeddings (and lm_head) to match the tokenizer.
# New rows are initialized from the existing embedding distribution.
model_4bit.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(32772, 4096)

In [ ]:
# Confirm the special-token list now includes <PAD> and the role markers.
tokenizer.all_special_tokens

['<s>', '</s>', '<unk>', '<PAD>', '<system>', '<user>', '<assistant>']

In [ ]:
# Beginning-of-sequence token.
tokenizer.bos_token

'<s>'

In [ ]:
# End-of-sequence token.
tokenizer.eos_token

'</s>'

## 7. Build a custom chat template

In [ ]:
# One raw dataset example (instruction / input / output).
dataset['train'][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [ ]:
# --- Define a CUSTOM chat template ---
# Hand-craft a template using our new role tokens, then format one example
# to preview the string the model will be trained on.
new_template = """<s><system>{system_prompt}</s><user>{user_prompt}</s><assistant>{model_answer}</s>"""

def format_dataset(example):

    system_prompt = example["instruction"]
    user_prompt = example["input"]
    model_answer = example["output"]

    formatted_text = new_template.format(

                                        system_prompt= system_prompt,
                                        user_prompt = user_prompt,
                                        model_answer =model_answer
    )

    return {"text": formatted_text}

format_dataset(dataset['train'][0])

{'text': "<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"}

In [ ]:
# Apply the custom template across the whole dataset.
dataset = dataset.map(format_dataset)
dataset['train']['text'][0]

Map:   0%|          | 0/2415 [00:00<?, ? examples/s]

"<s><system>translate this sentence in docker command</s><user>Give me a list of containers that have the Ubuntu image as their ancestor.</s><assistant>docker ps --filter 'ancestor=ubuntu'</s>"